# MScFE 620 Derivative Pricing - Group Work Project 2

## Team B: Jump Modeler

This is my Team B part of the group project. I worked on the Merton jump model questions and also included my Step 2 barrier questions:

- Q8: ATM European call and put when jump intensity is 0.75
- Q9: ATM European call and put when jump intensity is 0.25
- Q10: Delta and gamma for the options in Q8 and Q9
- Q14: Heston up-and-in call using the Q6 settings

In [22]:
import numpy as np
import pandas as pd

In [23]:
# Configs
S0 = 80.0
r = 0.055
sigma = 0.35
T = 0.25
K_ATM = 80.0

mu_jump = -0.5
delta_jump = 0.22

VANILLA_PATHS = 500_000
BARRIER_PATHS = 300_000
BARRIER_STEPS = 126
BUMP = 0.01 * S0
DISCOUNT = np.exp(-r * T)

In [24]:
def jump_compensator(mu_j=mu_jump, delta_j=delta_jump):
    """Expected proportional jump size in the Merton jump-diffusion model."""
    return np.exp(mu_j + 0.5 * delta_j**2) - 1.0


def merton_log_returns(
    jump_intensity,
    n_paths=VANILLA_PATHS,
    seed=620,
    maturity=T,
):
    """Simulate terminal log-returns under the risk-neutral Merton model."""
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n_paths)
    jump_counts = rng.poisson(jump_intensity * maturity, n_paths)
    jump_sums = rng.normal(
        mu_jump * jump_counts,
        delta_jump * np.sqrt(jump_counts),
    )
    kappa = jump_compensator()
    drift = (r - 0.5 * sigma**2 - jump_intensity * kappa) * maturity
    diffusion = sigma * np.sqrt(maturity) * z
    return drift + diffusion + jump_sums


def vanilla_price_from_log_returns(
    initial_stock,
    strike,
    option_type,
    log_returns,
):
    """Return discounted Monte Carlo option price and standard error."""
    terminal_stock = initial_stock * np.exp(log_returns)
    if option_type == "call":
        payoff = np.maximum(terminal_stock - strike, 0.0)
    elif option_type == "put":
        payoff = np.maximum(strike - terminal_stock, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    discounted_payoff = DISCOUNT * payoff
    price = discounted_payoff.mean()
    standard_error = discounted_payoff.std(ddof=1) / np.sqrt(len(discounted_payoff))
    return price, standard_error


def finite_difference_greeks(
    jump_intensity,
    strike,
    option_type,
    bump=BUMP,
):
    """Estimate delta and gamma using central finite differences."""
    seed = 12345 + int(jump_intensity * 100)
    log_returns = merton_log_returns(jump_intensity, seed=seed)
    price_down, _ = vanilla_price_from_log_returns(
        S0 - bump, strike, option_type, log_returns
    )
    price_mid, _ = vanilla_price_from_log_returns(
        S0, strike, option_type, log_returns
    )
    price_up, _ = vanilla_price_from_log_returns(
        S0 + bump, strike, option_type, log_returns
    )

    delta = (price_up - price_down) / (2 * bump)
    gamma = (price_up - 2 * price_mid + price_down) / (bump**2)
    return delta, gamma

## Questions 8
Using the Merton Model, price an ATM European call and an ATM European put with jump intensity parameter equal to 0.75.

## Questions 9
Using the Merton Model, price an ATM European call and an ATM European put with jump intensity parameter equal to 0.25.

In [25]:
vanilla_rows = []

for jump_intensity in [0.75, 0.25]:
    log_returns = merton_log_returns(
        jump_intensity,
        seed=620 + int(jump_intensity * 1000),
    )
    for option_type in ["call", "put"]:
        price, standard_error = vanilla_price_from_log_returns(
            S0,
            K_ATM,
            option_type,
            log_returns,
        )
        vanilla_rows.append(
            {
                "Question": 8 if jump_intensity == 0.75 else 9,
                "Jump intensity": jump_intensity,
                "Option": option_type.title(),
                "Strike": K_ATM,
                "Price": price,
                "Monte Carlo SE": standard_error,
            }
        )

vanilla_results = pd.DataFrame(vanilla_rows)
vanilla_results_display = vanilla_results.copy()
vanilla_results_display["Price"] = vanilla_results_display["Price"].round(2)
vanilla_results_display["Monte Carlo SE"] = vanilla_results_display[
    "Monte Carlo SE"
].round(4)
vanilla_results_display

,Question,Jump intensity,Option,Strike,Price,Monte Carlo SE
0,8,0.75,Call,80.0,8.33,0.0163
1,8,0.75,Put,80.0,7.20,0.0174
2,9,0.25,Call,80.0,6.85,0.0143
3,9,0.25,Put,80.0,5.72,0.0131


**Answer to Questions 8 and 9.** For jump intensity 0.75, I got an ATM call price of $8.33 and an ATM put price of $7.20. When I reduced the jump intensity to 0.25, the call price dropped to $6.85 and the put price dropped to $5.72.

So in this setup, more frequent jumps made both options more expensive. This makes sense because the terminal stock price has more spread when jumps are more likely, and options benefit from that extra uncertainty.

## Question 10
Calculate delta and gamma for each of the options in Questions 8 and 9.

In [26]:
greek_rows = []

for jump_intensity in [0.75, 0.25]:
    for option_type in ["call", "put"]:
        delta, gamma = finite_difference_greeks(
            jump_intensity,
            K_ATM,
            option_type,
        )
        greek_rows.append(
            {
                "Question": 10,
                "Jump intensity": jump_intensity,
                "Option": option_type.title(),
                "Delta": delta,
                "Gamma": gamma,
            }
        )

greek_results = pd.DataFrame(greek_rows)
greek_results_display = greek_results.copy()
greek_results_display["Delta"] = greek_results_display["Delta"].round(4)
greek_results_display["Gamma"] = greek_results_display["Gamma"].round(4)
greek_results_display

,Question,Jump intensity,Option,Delta,Gamma
0,10,0.75,Call,0.6491,0.0223
1,10,0.75,Put,-0.3509,0.0223
2,10,0.25,Call,0.5965,0.0258
3,10,0.25,Put,-0.4031,0.0258


**Answer to Question 10.** The call deltas are positive, which is expected because calls become more valuable when the stock price goes up. The put deltas are negative because puts lose value when the stock price goes up.

All the gammas are positive. This means the option price is curved with respect to the stock price, not just moving in a straight line. The call delta is higher for the 0.75 jump intensity case, while the put delta is less negative.

## Question 14
Using Heston model data from Question 6, price a European up-and-in call option (CUI) with a barrier level of $95 and a strike price of $95 as well. This CUI option becomes alive only if the stock price reaches the barrier level before maturity. Compare the price obtained to the one from the simple European call.

In [27]:
HESTON_V0 = 0.032
HESTON_KAPPA = 1.85
HESTON_THETA = 0.045
HESTON_XI = sigma
HESTON_RHO_Q6 = -0.70


def heston_up_and_in_call(
    initial_stock=S0,
    strike=95.0,
    barrier=95.0,
    v0=HESTON_V0,
    kappa=HESTON_KAPPA,
    theta=HESTON_THETA,
    xi=HESTON_XI,
    rho=HESTON_RHO_Q6,
    n_paths=BARRIER_PATHS,
    n_steps=BARRIER_STEPS,
    seed=62014,
):
    """Price a discretely monitored European up-and-in call under Heston."""
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    stock = np.full(n_paths, initial_stock, dtype=float)
    variance = np.full(n_paths, v0, dtype=float)
    barrier_hit = np.zeros(n_paths, dtype=bool)

    for _ in range(n_steps):
        z_stock = rng.standard_normal(n_paths)
        z_independent = rng.standard_normal(n_paths)
        z_variance = rho * z_stock + np.sqrt(1.0 - rho**2) * z_independent

        variance_positive = np.maximum(variance, 0.0)
        stock *= np.exp(
            (r - 0.5 * variance_positive) * dt
            + np.sqrt(variance_positive * dt) * z_stock
        )
        variance = np.maximum(
            variance
            + kappa * (theta - variance_positive) * dt
            + xi * np.sqrt(variance_positive * dt) * z_variance,
            0.0,
        )
        barrier_hit |= stock >= barrier

    simple_payoff = np.maximum(stock - strike, 0.0)
    cui_payoff = np.where(barrier_hit, simple_payoff, 0.0)

    simple_discounted = DISCOUNT * simple_payoff
    cui_discounted = DISCOUNT * cui_payoff

    return {
        "cui_price": cui_discounted.mean(),
        "cui_se": cui_discounted.std(ddof=1) / np.sqrt(n_paths),
        "simple_call_price": simple_discounted.mean(),
        "simple_call_se": simple_discounted.std(ddof=1) / np.sqrt(n_paths),
        "barrier_hit_rate": barrier_hit.mean(),
    }


q14 = heston_up_and_in_call()
heston_barrier_comparison = pd.DataFrame(
    [
        {
            "Question": 14,
            "Option": "Up-and-in call",
            "Correlation": HESTON_RHO_Q6,
            "Strike": 95.0,
            "Barrier": 95.0,
            "Price": q14["cui_price"],
            "Monte Carlo SE": q14["cui_se"],
            "Barrier hit rate": q14["barrier_hit_rate"],
        },
        {
            "Question": 14,
            "Option": "Simple European call",
            "Correlation": HESTON_RHO_Q6,
            "Strike": 95.0,
            "Barrier": np.nan,
            "Price": q14["simple_call_price"],
            "Monte Carlo SE": q14["simple_call_se"],
            "Barrier hit rate": np.nan,
        },
    ]
)

heston_barrier_display = heston_barrier_comparison.copy()
heston_barrier_display["Price"] = heston_barrier_display["Price"].round(2)
heston_barrier_display["Monte Carlo SE"] = heston_barrier_display[
    "Monte Carlo SE"
].round(4)
heston_barrier_display["Barrier hit rate"] = heston_barrier_display[
    "Barrier hit rate"
].round(4)
heston_barrier_display

,Question,Option,Correlation,Strike,Barrier,Price,Monte Carlo SE,Barrier hit rate
0,14,Up-and-in call,-0.7,95.0,95.0,0.03,0.0006,0.0231
1,14,Simple European call,-0.7,95.0,NaN,0.03,0.0006,NaN


**Answer to Question 14.** The up-and-in call price is about $0.03. The simple European call with the same $95 strike is also about $0.03.

The reason they are basically the same is that the barrier and strike are both $95. If the call finishes in the money, then the stock must have reached $95 at some point, so the option would already be knocked in. Only about 2.31% of the simulated paths touched the barrier.